# 02 — Grounding: GeoChat first, GeoGround/SAM fallback
**Owner: Person 2**

Priority 4 for the sprint. **Do not train a new grounding model immediately.** Strategy:

1. Try Person 1's GeoChat checkpoint for text-guided region grounding (prompt it to emit a bounding box, parse the coordinates out of its text output).
2. Evaluate IoU on VRSBench grounding samples.
3. Only if quality is insufficient, fall back to GeoGround or SAM (seeded from a rough GeoChat box/point) for a proper mask.

This keeps you independent from Person 1's fine-tuning timeline — you can start as soon as *any* GeoChat checkpoint (even the plain pretrained one) is loadable.

## 1. Environment

In [ ]:
# !pip install torch transformers pillow shapely --quiet
# !pip install segment-anything --quiet  # only if SAM fallback is needed
import sys, os
sys.path.insert(0, os.path.abspath('..'))


## 2. Load shared configuration

In [ ]:
from src.utils.io_utils import load_config

config = load_config('../configs/config.yaml')
grounding_cfg = config['models']['grounding']
vrsbench_cfg = config['datasets']['vrsbench']
grounding_cfg

## 3. Load GeoChat (reuse Person 1's checkpoint)
If Person 1's LoRA adapter isn't ready yet, start with the plain pretrained GeoChat checkpoint — grounding quality can be re-evaluated once the adapted version lands.

In [ ]:
# from transformers import AutoModel, AutoProcessor
# geochat = AutoModel.from_pretrained(config['models']['vqa']['checkpoint'])
# processor = AutoProcessor.from_pretrained(config['models']['vqa']['checkpoint'])


## 4. Prompt engineering for grounding
e.g. "Highlight the water body referred to in the query. Answer with a bounding box in the format [x1, y1, x2, y2]." Iterate on the exact phrasing — GeoChat's coordinate convention (normalized [0,1] vs pixel space) needs to be confirmed empirically.

In [ ]:
# sample_image = '<path to a sample RS image>'
# sample_query = 'Highlight the water body referred to in the query.'
# raw_output = ...  # run geochat inference here
# raw_output


## 5. Parse coordinates out of the model's text output
Handle both plausible formats (normalized vs pixel) and convert to a consistent pixel-space `[x1, y1, x2, y2]`.

In [ ]:
import re

def parse_bbox_from_text(text: str, image_width: int, image_height: int):
    """TODO(Person 2): robust parsing — handle normalized coords,
    different bracket/comma styles, and out-of-range values."""
    numbers = re.findall(r'[-+]?[0-9]*\.?[0-9]+', text)
    if len(numbers) < 4:
        return None
    x1, y1, x2, y2 = (float(n) for n in numbers[:4])
    if max(x1, y1, x2, y2) <= 1.0:  # looks normalized
        x1, x2 = x1 * image_width, x2 * image_width
        y1, y2 = y1 * image_height, y2 * image_height
    return [x1, y1, x2, y2]


## 6. Visualize the predicted box on the image

In [ ]:
# import matplotlib.pyplot as plt
# import matplotlib.patches as patches
# fig, ax = plt.subplots()
# ax.imshow(sample_image_array)
# x1, y1, x2, y2 = predicted_bbox
# ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='red'))
# plt.show()


## 7. Evaluate IoU on VRSBench grounding split

In [ ]:
from src.evaluation import metrics as M

# pred_boxes, gold_boxes = [], []  # collect from eval loop over VRSBench
# iou = M.grounding_iou(pred_boxes, gold_boxes)
# print('Mean IoU:', iou)


## 8. Decision point: is GeoChat grounding good enough?
If mean IoU is too low for a usable demo, fall back to GeoGround or SAM below. Otherwise, skip straight to Section 10 (export).

In [ ]:
GEOCHAT_GROUNDING_GOOD_ENOUGH = None  # TODO: set True/False after Section 7


## 9. Fallback: GeoGround or SAM (only if needed)
SAM needs a seed point/box — use GeoChat's rough output (Section 4) as the prompt, then let SAM refine it into a proper mask.

In [ ]:
# if not GEOCHAT_GROUNDING_GOOD_ENOUGH:
#     from segment_anything import sam_model_registry, SamPredictor
#     sam = sam_model_registry['vit_b'](checkpoint='<sam-checkpoint>')
#     predictor = SamPredictor(sam)
#     predictor.set_image(sample_image_array)
#     masks, scores, _ = predictor.predict(box=predicted_bbox)


## 10. Export the inference function
Move the stable implementation into `src/models/grounding_model.py`, setting `self.backbone` to whichever path ("GeoChat", "GeoGround", or "SAM") ended up being used.

In [ ]:
from src.common.schemas import RSModelResult, ResultMetadata, Evidence

# def predict(image, query):
#     bbox = ...
#     mask = ...  # None if not using SAM
#     return RSModelResult(
#         task='grounding', answer=f'Region located at {bbox}', confidence=...,
#         evidence=Evidence(bbox=bbox, mask=mask),
#         metadata=ResultMetadata(model='grounding_v1',
#                                  checkpoint=grounding_cfg['checkpoint'],
#                                  dataset='VRSBench',
#                                  backbone='GeoChat'),  # or 'GeoGround' / 'SAM'
#     ).to_dict()
